# 최소제곱법과 회귀

> 선형대수 10강 · 직교성과 최소제곱법

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [최소제곱법과 회귀](https://mioon1402.github.io/timeseriesdata/linalg/L10-least-squares.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.**

In [ ]:

print('준비 완료')

## 0. 기초 다지기 — 왜 해가 없는가

## 1. "가장 가까운" 을 정의하기

## 2. 정규방정식

## 3. 직선 맞추기

## 4. 왜 하필 제곱인가

## 5. numpy 로 확인하기

**10-1. 해가 없는 것부터 확인**

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

t = np.array([0., 1., 2., 3.])
y = np.array([1., 2., 2., 5.])

A = np.column_stack([t, np.ones_like(t)])   # [t | 1] — 기울기와 절편
print("A =")
print(A)
print("b =", y)
print()
print("rank(A)      =", np.linalg.matrix_rank(A))
print("rank([A|b])  =", np.linalg.matrix_rank(np.column_stack([A, y])))
print("→ 랭크가 다르다 = b 가 열공간 밖 = 해가 없다 (6강)")

**10-2. 정규방정식 직접 풀기**

In [ ]:
AtA = A.T @ A
Atb = A.T @ y

print("AᵀA =")
print(AtA, "  ← 2×2 로 줄었고 대칭이다")
print("\nAᵀb =", Atb)
print()

x_hat = np.linalg.solve(AtA, Atb)           # inv 대신 solve (4강)
print("x̂ = (기울기 c, 절편 d) =", x_hat)
print(f"→ 직선:  y = {x_hat[0]:.2f} t + {x_hat[1]:.2f}")

**10-3. 라이브러리와 비교**

In [ ]:
ls, 잔차합, 랭크, 특이값 = np.linalg.lstsq(A, y, rcond=None)
print("lstsq   :", ls)
print("polyfit :", np.polyfit(t, y, 1))
print("직접    :", x_hat)
print()
print("전부 같은가:", np.allclose(ls, x_hat) and np.allclose(np.polyfit(t, y, 1), x_hat))
print()
print("※ 실무에서는 lstsq 를 쓰세요. 정규방정식을 직접 풀면")
print("   AᵀA 를 만드는 과정에서 조건수가 제곱이 되어 오차가 커집니다 (§6).")

**10-4. 회귀는 정말 투영인가**

In [ ]:
p = A @ x_hat          # 예측값 = 그림자
e = y - p              # 잔차   = 오차

print("예측 p =", p)
print("잔차 e =", e)
print()
print("Aᵀe =", A.T @ e, "  ← 0. 잔차가 열공간과 직교 (9강)")
print("  첫째 성분 = Σ tᵢeᵢ = 0")
print("  둘째 성분 = Σ eᵢ   = 0   ← 절편이 있으면 잔차 합이 0")
print()

P = A @ np.linalg.inv(A.T @ A) @ A.T
print("투영행렬로 계산한 예측 =", P @ y)
print("같은가:", np.allclose(P @ y, p))
print("P² = P :", np.allclose(P @ P, P), "   trace(P) =", round(np.trace(P), 6), "= 열 개수")

**10-5. R² 는 피타고라스다**

In [ ]:
y_bar = y.mean()
SST = ((y - y_bar) ** 2).sum()      # 총 변동
SSR = ((p - y_bar) ** 2).sum()      # 설명된 변동
SSE = (e ** 2).sum()                # 남은 변동

print(f"SST (총)     = {SST:.4f}")
print(f"SSR (설명)   = {SSR:.4f}")
print(f"SSE (잔차)   = {SSE:.4f}")
print(f"SSR + SSE    = {SSR + SSE:.4f}   ← SST 와 같다")
print()
print(f"R² = SSR/SST = {SSR/SST:.4f}  = 1 - SSE/SST = {1 - SSE/SST:.4f}")
print()
print("이 분해가 성립하는 이유는 (p - ȳ) ⟂ e 이기 때문 — 피타고라스 정리(8강)")
print("직교 확인:", round((p - y_bar) @ e, 12))

**10-6. 곡선도 '선형' 회귀로 맞춘다**

In [ ]:
# y = a + b·t + c·t²  는 미지수 a,b,c 에 대해서는 여전히 '선형' 이다
t2 = np.linspace(0, 4, 9)
y2 = 2 - 1.5 * t2 + 0.6 * t2 ** 2 + np.random.default_rng(0).normal(0, 0.3, size=9)

A2 = np.column_stack([np.ones_like(t2), t2, t2 ** 2])   # [1 | t | t²]
계수, *_ = np.linalg.lstsq(A2, y2, rcond=None)

print("설계행렬 A2 의 처음 3행 =")
print(A2[:3])
print()
print(f"추정: y = {계수[0]:.2f} + {계수[1]:.2f} t + {계수[2]:.2f} t²")
print(f"참값: y = 2.00 + -1.50 t + 0.60 t²")
print()
print("→ '선형회귀' 의 '선형' 은 t 가 아니라 '계수' 에 대한 것이다.")
print("   열을 무엇으로 채우든(t², log t, sin t) 방법은 똑같다.")

**10-7. 이상치의 영향**

In [ ]:
y3 = y.copy()
print("원래  :", np.linalg.lstsq(A, y3, rcond=None)[0])

y3[2] = 12.0                          # 점 하나를 크게 흔든다
print("이상치:", np.linalg.lstsq(A, y3, rcond=None)[0])
print()
print("→ 점 하나가 기울기와 절편을 통째로 끌고 갔다.")
print("   제곱이 큰 오차에 큰 발언권을 주기 때문이다.")

**10-8. 연습문제**

In [ ]:
# 문제 1. 점 (1,1), (2,3), (3,4) 에 최소제곱 직선을 맞춰보세요.

# 문제 2. 그 결과의 잔차를 구하고, 합이 0인지 Aᵀe = 0 인지 확인하세요.

# 문제 3. 절편 없이(원점을 지나는 직선 y = ct) 맞추면 잔차 합이 0일까요?
#         직접 확인해보세요.

# 아래에 직접 써보세요

**모범 답안**

In [ ]:
tt = np.array([1., 2., 3.]); yy = np.array([1., 3., 4.])

# 문제 1
AA = np.column_stack([tt, np.ones_like(tt)])
ans, *_ = np.linalg.lstsq(AA, yy, rcond=None)
print(f"문제 1: y = {ans[0]:.3f} t + {ans[1]:.3f}")

# 문제 2
ee = yy - AA @ ans
print(f"문제 2: 잔차 {np.round(ee, 6)}  합 {ee.sum():.2e}  Aᵀe {np.round(AA.T @ ee, 9)}")

# 문제 3 — 절편 열이 없으면 '잔차 합 = 0' 이 보장되지 않는다
AA0 = tt.reshape(-1, 1)
ans0, *_ = np.linalg.lstsq(AA0, yy, rcond=None)
ee0 = yy - AA0 @ ans0
print(f"\n문제 3: y = {ans0[0]:.3f} t (절편 없음)")
print(f"        잔차 {np.round(ee0, 4)}  합 {ee0.sum():.4f}  ← 0 이 아니다")
print("        Aᵀe =", np.round(AA0.T @ ee0, 9), " ← 이건 여전히 0")
print("        '잔차 합 = 0' 은 절편 열(전부 1)이 있을 때만 나오는 결과다.")

## 6. 실무: 언제 무너지는가

---

전체 강의 목록 → [눈으로 보는 수학·통계](https://mioon1402.github.io/timeseriesdata/)